# 02. 서비스 시뮬레이션 데모

저장된 모델을 불러와 배차 → 구간별 에너지 예측 → SOC → 충전 계획 → ETA를
한 번에 실행한다. 모델 학습은 하지 않는다 (먼저 `python scripts/train_model.py` 실행).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
from ev_logistics.simulation import run_service_simulation

## 1. 한 건 실행

In [ ]:
result = run_service_simulation('12345678', '2026-08-28')
d = result['dispatch']
print(f"차량 {result['vehicle']['display_name']} · 화물 {result['payload_kg']:.0f}kg · 출발 SOC {d['battery_soc_pct']:.0f}%")
print(f"날씨 {d['weather_type']} {d['ambient_temp_C']}C · HVAC {d['hvac_mode']} {d['hvac_power_kw']}kW")
print(f"예상 소비량 {result['total_energy_kwh']:.1f} kWh ({result['predicted_kwh_per_100km']:.2f} kWh/100km)")
print(f"모델 R2 {result['metrics']['R2']:.4f}")

## 2. 구간별 예측

In [ ]:
result['segments'][['sequence', 'point_name', 'segment_distance_km',
                    'segment_average_grade_pct', 'predicted_kwh_per_100km',
                    'predicted_energy_kwh']]

## 3. SOC·충전·ETA

In [ ]:
sim = result['simulation']
print('운행 가능:', sim['feasible'])
if sim['feasible']:
    print(f"도착 {sim['arrival_time']:%Y-%m-%d %H:%M} · 총 {sim['total_minutes']:.0f}분")
display(pd.DataFrame(sim['charges']))
display(pd.DataFrame(sim['legs']))

## 4. 여러 사원번호 비교

In [ ]:
summary = []
for emp in ['12345678', '20260828', '00000042', '99887766']:
    r = run_service_simulation(emp, '2026-08-28')
    s = r['simulation']
    summary.append({
        '사원번호': emp,
        '차량': r['vehicle']['display_name'],
        '출발SOC': r['dispatch']['battery_soc_pct'],
        '소비량_kWh': round(r['total_energy_kwh'], 1),
        '충전횟수': len(s['charges']),
        '운행가능': s['feasible'],
        '총분': round(s['total_minutes']) if s['feasible'] else None,
    })
pd.DataFrame(summary)